# 🛣️ Crack Segmentation Image Pipeline
### YOLOv8x-seg (HuggingFace) + Skeletonization

---

**Features:**
- ✅ Auto-downloads YOLOv8x-seg from HuggingFace (~89% mAP50)
- ✅ Image folder input (no video required)
- ✅ Skeletonization for crack length measurement
- ✅ GPS optional
- ✅ Masked/Unmasked output folders

**Model Source**: [OpenSistemas/YOLOv8-crack-seg](https://huggingface.co/OpenSistemas/YOLOv8-crack-seg)

In [ ]:
# @title 1️⃣ Setup & Configuration
import torch
import os

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Enable GPU: Runtime → Change runtime type → GPU")

# @markdown ### ⚙️ Features
USE_GPS = False  # @param {type:"boolean"}
USE_HUGGINGFACE_MODEL = True  # @param {type:"boolean"}
# @markdown *If True, downloads YOLOv8x-seg from HuggingFace. If False, uses custom model path.*

# @markdown ### 🔧 Inference Settings
CONFIDENCE_THRESHOLD = 0.25  # @param {type:"number"}

# @markdown ### 📏 Calibration
PIXELS_PER_METER = 500.0  # @param {type:"number"}

# @markdown ### 🖼️ Image Resizing
MAX_IMAGE_DIM = 0  # @param {type:"integer"}
# @markdown *Set to 0 to disable. Recommended: 1024 for large images*

print(f"✅ Settings Loaded")
print(f"   HuggingFace Model: {'Enabled' if USE_HUGGINGFACE_MODEL else 'Disabled'}")

!pip install -q ultralytics opencv-python pandas numpy tqdm scikit-image huggingface_hub

In [ ]:
# @title 2️⃣ Download HuggingFace Model OR Define Custom Path
from huggingface_hub import hf_hub_download

# @markdown ### 📁 Input Images Folder
IMAGES_FOLDER = "/content/images/"  # @param {type:"string"}

# @markdown ### 🧠 Custom Model Path (if not using HuggingFace)
CUSTOM_MODEL_PATH = "/content/crack_seg_best.pt"  # @param {type:"string"}

# @markdown ### 📍 Optional GPS File
GPS_LOG_PATH = "/content/gps_log.csv"  # @param {type:"string"}

if USE_HUGGINGFACE_MODEL:
    print("🔄 Downloading YOLOv8x-seg from HuggingFace...")
    try:
        SEG_MODEL_PATH = hf_hub_download(
            repo_id="OpenSistemas/YOLOv8-crack-seg",
            filename="yolov8x-seg.pt"
        )
        print(f"✅ Model downloaded: {SEG_MODEL_PATH}")
    except Exception as e:
        print(f"⚠️ HuggingFace download failed: {e}")
        print("   Trying alternative filename...")
        try:
            SEG_MODEL_PATH = hf_hub_download(
                repo_id="OpenSistemas/YOLOv8-crack-seg",
                filename="best.pt"
            )
            print(f"✅ Model downloaded: {SEG_MODEL_PATH}")
        except:
            print("❌ Could not download. Using custom path.")
            SEG_MODEL_PATH = CUSTOM_MODEL_PATH
else:
    SEG_MODEL_PATH = CUSTOM_MODEL_PATH
    print(f"📂 Using custom model: {SEG_MODEL_PATH}")

# Check images folder
import glob
if os.path.exists(IMAGES_FOLDER):
    images = glob.glob(os.path.join(IMAGES_FOLDER, "*.jpg")) + \
             glob.glob(os.path.join(IMAGES_FOLDER, "*.jpeg")) + \
             glob.glob(os.path.join(IMAGES_FOLDER, "*.png"))
    print(f"✅ Found {len(images)} images in {IMAGES_FOLDER}")
else:
    print(f"⚠️ Create folder and upload images: {IMAGES_FOLDER}")

In [ ]:
# @title 3️⃣ Load GPS Data (Optional)
import pandas as pd

gps_data = None

if USE_GPS and os.path.exists(GPS_LOG_PATH):
    try:
        df = pd.read_csv(GPS_LOG_PATH)
        df.columns = df.columns.str.strip().str.lower()
        if 'time' in df.columns:
            df['time'] = pd.to_datetime(df['time'], format='mixed', utc=True)
            df = df.sort_values('time')
        gps_data = df
        print(f"✅ Loaded {len(gps_data)} GPS points")
    except Exception as e:
        print(f"⚠️ Error loading GPS: {e}")
else:
    print("⏩ GPS Skipped")

In [ ]:
# @title 4️⃣ Load Segmentation Model
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if os.path.exists(SEG_MODEL_PATH):
    seg_model = YOLO(SEG_MODEL_PATH)
    seg_model.to(device)
    SEG_CLASSES = seg_model.names
    print(f"✅ Model Loaded: {SEG_MODEL_PATH}")
    print(f"   Classes: {SEG_CLASSES}")
else:
    print(f"❌ Model NOT found: {SEG_MODEL_PATH}")

In [ ]:
# @title 5️⃣ Core Functions
import cv2
import numpy as np
from skimage.morphology import skeletonize

def resize_if_needed(image, max_dim):
    if max_dim <= 0:
        return image, 1.0
    h, w = image.shape[:2]
    if max(h, w) <= max_dim:
        return image, 1.0
    scale = max_dim / max(h, w)
    new_size = (int(w * scale), int(h * scale))
    return cv2.resize(image, new_size), scale

def measure_crack_length(mask, ppm):
    """Measure crack length using skeletonization."""
    binary_mask = mask > 0
    skeleton = skeletonize(binary_mask)
    pixel_length = np.sum(skeleton)
    
    if pixel_length == 0:
        contours, _ = cv2.findContours(
            binary_mask.astype(np.uint8) * 255,
            cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours:
            pixel_length = sum(cv2.arcLength(c, True) for c in contours) / 2
    
    length_m = pixel_length / ppm
    return max(length_m, 0.01) if pixel_length > 0 else 0.0, skeleton, binary_mask

def measure_area(mask, ppm):
    pixel_area = np.sum(mask > 0)
    area_m2 = pixel_area / (ppm ** 2)
    return max(area_m2, 0.001) if pixel_area > 0 else 0.0

print("✅ Functions ready")

In [ ]:
# @title 6️⃣ Run Image Pipeline
import glob
import shutil
from tqdm import tqdm

def run_image_pipeline():
    image_files = glob.glob(os.path.join(IMAGES_FOLDER, "*.jpg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.jpeg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.png"))
    
    if not image_files:
        print(f"❌ No images found in {IMAGES_FOLDER}")
        return pd.DataFrame()
    
    print(f"\n{'='*60}")
    print(f"🚀 CRACK SEGMENTATION PIPELINE")
    print(f"{'='*60}")
    print(f"✅ {len(image_files)} images")
    print(f"🧠 Model: {os.path.basename(SEG_MODEL_PATH)}")
    print(f"📏 Scale: {PIXELS_PER_METER} px/m")
    
    os.makedirs("results/unmasked", exist_ok=True)
    os.makedirs("results/masked", exist_ok=True)
    
    all_records = []
    
    for img_path in tqdm(image_files, desc="Processing"):
        filename = os.path.basename(img_path)
        
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        
        frame, resize_scale = resize_if_needed(frame, MAX_IMAGE_DIM)
        adjusted_ppm = PIXELS_PER_METER * resize_scale
        height, width = frame.shape[:2]
        
        results = seg_model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        
        if not results or results[0].masks is None:
            continue
        
        masks = results[0].masks.data.cpu().numpy()
        boxes = results[0].boxes
        
        for idx, (mask, box) in enumerate(zip(masks, boxes)):
            class_id = int(box.cls[0])
            class_name = SEG_CLASSES.get(class_id, 'crack')
            confidence = float(box.conf[0])
            
            mask_resized = cv2.resize(mask, (width, height)) > 0.5
            
            # All detections treated as cracks for this model
            length_m, skeleton, binary_mask = measure_crack_length(mask_resized, adjusted_ppm)
            measurement = min(length_m, 10.0)
            label = f"{class_name} | {measurement:.2f}m"
            
            out_filename = f"{class_name}_{idx}_{filename}"
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            # UNMASKED
            unmasked = frame.copy()
            cv2.rectangle(unmasked, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(unmasked, label, (x1, max(y1-10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.imwrite(f"results/unmasked/{out_filename}", unmasked)
            
            # MASKED
            masked = frame.copy()
            overlay = np.zeros_like(masked)
            overlay[binary_mask] = (200, 200, 0)  # Cyan
            overlay[skeleton] = (0, 255, 255)      # Yellow skeleton
            masked = cv2.addWeighted(masked, 0.7, overlay, 0.3, 0)
            cv2.rectangle(masked, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(masked, label, (x1, max(y1-10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.imwrite(f"results/masked/{out_filename}", masked)
            
            record = {
                'source_image': filename,
                'detection_idx': idx,
                'class': class_name,
                'confidence': round(confidence, 4),
                'length_m': round(measurement, 4),
                'image_path': out_filename
            }
            
            if gps_data is not None and len(gps_data) > 0:
                record['latitude'] = gps_data.iloc[0].get('latitude', None)
                record['longitude'] = gps_data.iloc[0].get('longitude', None)
            
            all_records.append(record)
    
    print(f"\n{'='*60}")
    print(f"📊 PIPELINE COMPLETE")
    print(f"{'='*60}")
    
    if all_records:
        df = pd.DataFrame(all_records)
        df.to_csv("results/detections.csv", index=False)
        
        print(f"✅ Detections: {len(df)}")
        print(f"   - Unmasked: {len(os.listdir('results/unmasked'))} files")
        print(f"   - Masked: {len(os.listdir('results/masked'))} files")
        
        shutil.make_archive("crack_seg_results", 'zip', "results")
        print(f"\n📦 crack_seg_results.zip created")
        
        try:
            from google.colab import files
            files.download("crack_seg_results.zip")
        except:
            pass
        
        return df
    else:
        print("⚠️ No detections found.")
        return pd.DataFrame()

df = run_image_pipeline()
df